In [ ]:
import pandas as pd
import nltk

In [ ]:
df = pd.read_csv('/content/samba_montclair.csv')

In [ ]:
df.head()

,userName,address,rating,dateOfReview,reviewDesc
0,Roger A.,NaN,5,10/31/2021,I have been to Montclair a couple of times but...
1,Yuribel A.,NaN,4,2/26/2022,"I have been here twice, the first time with my..."
2,Katie M.,NaN,4,2/24/2022,I was a bit skeptical at first as i wasnt sure...
3,Lisa M.,NaN,3,2/12/2022,I was SO excited to come here and eat after it...
4,Maria S.,NaN,4,2/12/2022,Great food and ambiance! Parking is difficult ...


**This allows us to see which reviews are positive or negitive**

In [ ]:
df['sentiment'] = df['rating'].apply(lambda x: 'neg' if x in [1, 2, 3] else 'pos')

df.head()

,userName,address,rating,dateOfReview,reviewDesc,sentiment
0,Roger A.,NaN,5,10/31/2021,I have been to Montclair a couple of times but...,pos
1,Yuribel A.,NaN,4,2/26/2022,"I have been here twice, the first time with my...",pos
2,Katie M.,NaN,4,2/24/2022,I was a bit skeptical at first as i wasnt sure...,pos
3,Lisa M.,NaN,3,2/12/2022,I was SO excited to come here and eat after it...,neg
4,Maria S.,NaN,4,2/12/2022,Great food and ambiance! Parking is difficult ...,pos


**Here we are cleaning the data by getting rid of puncuation, removing stopwords and coverting all words to lower case**

In [ ]:
df['reviewDesc'] = df['reviewDesc'].str.lower()

In [ ]:
import string
df['reviewDesc'] = df['reviewDesc'].apply(lambda x: ''.join([char for char in x if char not in string.punctuation]))

In [ ]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('stopwords')

def clean_text(text):
    tokens = word_tokenize(text)
    stop_words = set(stopwords.words('english'))
    text = ' '.join([word for word in tokens if word not in stop_words])
    return text

df['reviewDesc'] = df['reviewDesc'].apply(clean_text)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
df.head()

,userName,address,rating,dateOfReview,reviewDesc,sentiment
0,Roger A.,NaN,5,10/31/2021,montclair couple times somehow missed place se...,pos
1,Yuribel A.,NaN,4,2/26/2022,twice first time friend dinner 2nd time boyfri...,pos
2,Katie M.,NaN,4,2/24/2022,bit skeptical first wasnt sure would like food...,pos
3,Lisa M.,NaN,3,2/12/2022,excited come eat recommended end really truly ...,neg
4,Maria S.,NaN,4,2/12/2022,great food ambiance parking difficult customer...,pos


**We can use this split the review text from the postives and negitives**

In [ ]:
import random

reviews = [(text.split(), sentiment) for text, sentiment in zip(df['reviewDesc'], df['sentiment'])]

random.shuffle(reviews)

In [ ]:
for i in range(5):
    print(reviews[i])

print(reviews[:5])

(['finally', 'got', 'try', 'place', 'excited', 'never', 'brazilian', 'food', 'looking', 'forward', 'arrived', 'usual', 'little', 'trouble', 'finding', 'parking', 'eventually', 'walked', 'told', 'server', 'luis', 'reservations', 'allowed', 'pick', 'wanted', 'sit', 'fyi', 'liked', 'decor', 'restaurant', 'byob', 'luis', 'server', 'ok', 'service', 'could', 'better', 'wasnt', 'terrible', 'werent', 'attentive', 'would', 'liked', 'keep', 'getting', 'servers', 'attention', 'come', 'brunch', 'angus', 'burger', 'homemade', 'bread', 'friend', 'egg', 'cheese', 'sandwich', 'detox', 'drink', 'remind', 'drink', 'forgot', 'give', 'really', 'liked', 'food', 'burger', 'tasted', 'fresh', 'fried', 'yucca', 'good', 'well', 'friend', 'enjoyed', 'sandwich', 'well', 'said', 'detox', 'drink', 'good', 'cost', 'little', '37', 'wasnt', 'bad', 'good', 'experience'], 'neg')
(['went', 'friend', 'weekday', 'restaurant', 'try', 'list', 'good', 'ambiance', 'amazing', 'definitely', 'day', 'night', 'special', 'occasion',

**Here we prepare for the model by making word features for each one of the reviews and their words.**

In [ ]:
all_words = nltk.FreqDist(w.lower() for row in df['reviewDesc'] for w in row.split())
word_features = list(all_words)

def document_features(document):
    document_words = set(document)
    features = {}
    for word in word_features:
        features['contains({})'.format(word)] = (word in document_words)
    return features

**Here we split the data 70 percent and train it and test it with the other 30 percent as a predictive model in order to see what are the top features/words used in reviews that inform us if the reviews going to be positive or negitive.**

In [ ]:
split = 0.7

featuresets = [(document_features(d), c) for (d,c) in reviews]

train_set, test_set = featuresets[int(round(len(featuresets)*split,0))-1:], featuresets[:int(round(len(featuresets)*split,0))-1]

nb_classifier = nltk.NaiveBayesClassifier.train(train_set)

**This was the best accuracy i could acheive**

In [ ]:
print(nltk.classify.accuracy(nb_classifier, test_set))

0.8027210884353742


In [ ]:
nb_classifier.show_most_informative_features(n=5)

Most Informative Features
       contains(another) = True              neg : pos    =     16.6 : 1.0
          contains(used) = True              neg : pos    =     14.0 : 1.0
         contains(asked) = True              neg : pos    =     13.0 : 1.0
       contains(arrived) = True              neg : pos    =     11.5 : 1.0
          contains(bill) = True              neg : pos    =     11.5 : 1.0


In [ ]:
from nltk.classify import DecisionTreeClassifier
train_feats = train_set
test_feats = test_set

dt_classifier = DecisionTreeClassifier.train(train_feats,
                                             binary=True,
                                             entropy_cutoff=0.8,
                                             depth_cutoff=5,
                                             support_cutoff=30)

**NAIVE BAYES RESULTS:**

In [ ]:
import collections

from nltk.metrics.scores import (accuracy, precision, recall, f_measure)

print('Accuracy:', nltk.classify.accuracy(nb_classifier, test_set))

refsets = collections.defaultdict(set)
testsets = collections.defaultdict(set)

for i, (feats, label) in enumerate(test_set):
    refsets[label].add(i)
    observed = nb_classifier.classify(feats)
    testsets[observed].add(i)

print('Positive Precision:', precision(refsets['pos'], testsets['pos']))
print('Positive Recall:', recall(refsets['pos'], testsets['pos']))
print('Positive F1-measure:', f_measure(refsets['pos'], testsets['pos']))
print('Negative Precision:', precision(refsets['neg'], testsets['neg']))
print('Negative Recall:', recall(refsets['neg'], testsets['neg']))
print('Negative F1-measure:', f_measure(refsets['neg'], testsets['neg']))

from sklearn.metrics import roc_auc_score
import numpy as np

y_true = [1 if label == 'pos' else 0 for (_, label) in test_set]
y_scores = np.random.rand(len(test_set))

auc = roc_auc_score(y_true, y_scores)
print("AUC:", auc)


Accuracy: 0.8027210884353742
Positive Precision: 0.8728813559322034
Positive Recall: 0.8803418803418803
Positive F1-measure: 0.8765957446808511
Negative Precision: 0.5172413793103449
Negative Recall: 0.5
Negative F1-measure: 0.5084745762711864
AUC: 0.5735992402659069


**DECISION TREE RESULTS:**

In [ ]:
refsets_dt = collections.defaultdict(set)
testsets_dt = collections.defaultdict(set)

y_pred_dt = []
y_true_dt = []

for i, (feats, label) in enumerate(test_feats):
    refsets_dt[label].add(i)
    observed = dt_classifier.classify(feats)
    testsets_dt[observed].add(i)

    y_pred_dt.append(1 if observed == 'pos' else 0)
    y_true_dt.append(1 if label == 'pos' else 0)


print('Decision Tree pos precision:', precision(refsets_dt['pos'], testsets_dt['pos']))
print('Decision Tree pos recall:', recall(refsets_dt['pos'], testsets_dt['pos']))
print('Decision Tree pos F-measure:', f_measure(refsets_dt['pos'], testsets_dt['pos']))
print('Decision Tree neg precision:', precision(refsets_dt['neg'], testsets_dt['neg']))
print('Decision Tree neg recall:', recall(refsets_dt['neg'], testsets_dt['neg']))
print('Decision Tree neg F-measure:', f_measure(refsets_dt['neg'], testsets_dt['neg']))

accuracy_dt = nltk.classify.accuracy(dt_classifier, test_feats)
print('Decision Tree Accuracy:', accuracy_dt)

from sklearn.metrics import roc_auc_score
auc_dt = roc_auc_score(y_true_dt, y_pred_dt)
print("Decision Tree AUC:", auc_dt)

Decision Tree pos precision: 0.8099762470308789
Decision Tree pos recall: 0.9715099715099715
Decision Tree pos F-measure: 0.8834196891191711
Decision Tree neg precision: 0.5
Decision Tree neg recall: 0.1111111111111111
Decision Tree neg F-measure: 0.18181818181818182
Decision Tree Accuracy: 0.7959183673469388
Decision Tree AUC: 0.5413105413105413


**In general, the naive bayes did better then the decision tree model. We can assume that it did better because of the ROC AUC as the naive bayes was at .57 and the decision tree was at .54 but there are other reasons as well. While the decision tree positive recall was amazing the negitive recall which shows how many true positives it gotten out of all the whole model but the neg recall was .11 which is almost negitive which is the reason why the deicison tree model did worse then the naive bayes. I would choose the naive bayes over the decision tree.**